# Week 4 Exercise: Test Generation (Python)

In [ ]:
import os
import re

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openrouter_key = os.getenv("OPEN_ROUTER_API_KEY")


if openrouter_key:
    print(f"OpenRouter key present (begins {openrouter_key[:8]}...)")
else:
    print("Set OPEN_ROUTER_API_KEY in .env to use the app.")

In [ ]:

openrouter_url = "https://openrouter.ai/api/v1"
openai = OpenAI(api_key=openrouter_key, base_url=openrouter_url) if openrouter_key else None

MODELS = {
    "OpenAI GPT-4o": (openai, "openai/gpt-4o"),
    "OpenAI GPT-4o-mini": (openai, "openai/gpt-4o-mini"),
    "OpenAI GPT-4o-mini (latest)": (openai, "openai/gpt-4o-mini-2024-07-18"),
    "Anthropic Claude 3.5 Sonnet": (openai, "anthropic/claude-3.5-sonnet"),
    "Anthropic Claude 3.5 Haiku": (openai, "anthropic/claude-3.5-haiku"),
    "Anthropic Claude 3 Opus": (openai, "anthropic/claude-3-opus"),
} if openai else {}

print("Models:", list(MODELS.keys()) or "None (set OPEN_ROUTER_API_KEY)")

In [ ]:
SYSTEM_PROMPT = """You are a Python engineer. Your task is to write Python tests for the given Python code.
Rules:
- Output raw Python only: first the original code (the module under test), then test functions (def test_...).
- Do not wrap the output in markdown or ``` blocks.
- Test every public function and class method with at least one simple test that should pass.
- Use assert for checks. No pytest required; plain Python with assert is fine.
"""

def user_prompt_for(python_source: str) -> str:
    return f"""Write Python tests for the following code. Include the original code at the top, then the test functions. Output only executable Python (no markdown, no ```).

Python code to test:

{python_source}
"""

In [ ]:
def messages_for(python_source: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt_for(python_source)},
    ]

In [ ]:
def extract_python(reply: str) -> str:
    """Remove markdown code fences if present; return raw Python."""
    m = re.search(r"```(?:python)?\s*\n?(.*?)```", reply, re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return reply.strip()


def generate_tests(model_name: str, python_source: str) -> str:
    if model_name not in MODELS:
        return "# Error: unknown model"
    client, model_id = MODELS[model_name]
    response = client.chat.completions.create(
        model=model_id,
        messages=messages_for(python_source),
        max_tokens=4096,
    )
    reply = (response.choices[0].message.content or "").strip()
    return extract_python(reply)

In [ ]:
def ui_generate(model_name: str, python_source: str) -> str:
    if not (python_source and python_source.strip()):
        return "(Enter Python code to test)"
    if not MODELS or model_name not in MODELS:
        return "(No model selected or set OPEN_ROUTER_API_KEY in .env)"
    return generate_tests(model_name, python_source.strip())


with gr.Blocks(title="Test Generation") as demo:
    gr.Markdown("## Generate Python tests from your code")
    gr.Markdown("Paste your Python code, pick a model, then click **Generate tests**.")

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=list(MODELS.keys()) if MODELS else ["(Set OPEN_ROUTER_API_KEY first)"],
            value=list(MODELS.keys())[0] if MODELS else None,
            label="Model",
        )
    python_input = gr.Textbox(
        label="Python code to test",
        lines=12,
        placeholder="def add(a, b): return a + b\n\ndef is_even(n): return n % 2 == 0",
    )
    run_btn = gr.Button("Generate tests")
    generated_code = gr.Textbox(label="Generated test code", lines=14)
    run_btn.click(
        fn=ui_generate,
        inputs=[model_dropdown, python_input],
        outputs=generated_code,
    )

demo.launch(share=False)